In [ ]:
!pip install pycaret

In [18]:
import pandas as pd
import numpy as np
import tensorflow as tf
from pycaret.classification import setup, compare_models

from pathlib import Path

from IPython.display import clear_output

## Leitura

In [ ]:
########################################
# Download dos dados
########################################
# Extrai os arquivos para 
# a pasta /content/dataset
########################################

%%bash --out output --err error
set -euxo pipefail

if [ ! -d dataset ]; then
    mkdir -p "/root/.kaggle"
    cp -a "/content/drive/My Drive/<PASTA_DO_PROJETO>/kaggle.json" "/root/.kaggle"
    chmod 600 "/root/.kaggle/kaggle.json"

    pip install kaggle==1.5.6
    kaggle datasets download -d "paultimothymooney/kermany2018" -p "/content"

    unzip "/content/kermany2018.zip" "OCT2017 /*" -d "/content"
    mv "OCT2017 " "dataset"
fi

In [8]:
########################################
# Leitura dos dados
########################################
# split_paper.csv
# ---
# Divide as bases
# de acordo com o artigo,
# logo unindo validação ao teste.
# Sem exclusão de pacientes duplicados.
########################################
# split_baseline.csv
# ---
# Divide as imagens
# unindo validação ao treino,
# sem excluir pacientes contidos
# no conjunto de (treino ∪ val) ∩ teste.
########################################
# split_honest.csv
# ---
# Divide os dados
# unindo validação ao treino,
# excluindo pacientes contidos
# no conjunto de (treino ∪ val) ∩ teste.
# É a que serve como comparação 
# direta com o outro grupo de OCT.
########################################

df = pd.read_csv("/content/drive/My Drive/<PASTA_DO_PROJETO>/Notebooks_Vinicius/Splits/split_honest.csv")

datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0/255,
)

train_generator = datagen.flow_from_dataframe(
    df.query('new_split == "train"'),
    target_size=(512, 512),
    x_col='path',
    y_col='file_label',
    class_mode='categorical',
    batch_size=128,
)

val_generator = datagen.flow_from_dataframe(
    df.query('new_split == "val"'),
    target_size=(512, 512),
    x_col='path',
    y_col='file_label',
    class_mode='categorical',
    batch_size=128,
)

test_generator = datagen.flow_from_dataframe(
    df.query('new_split == "test"'),
    target_size=(512, 512),
    x_col='path',
    y_col='file_label',
    class_mode='categorical',
    batch_size=128,
)

Found 0 validated image filenames belonging to 0 classes.
Found 0 validated image filenames belonging to 0 classes.
Found 0 validated image filenames belonging to 0 classes.


## Experimentos

### Utilidades

In [19]:
rootpath = Path("/content/drive/My Drive/<PASTA_DO_PROJETO>/Notebooks_Vinicius/Features/")


def load_features(filename):
    data = np.load(rootpath / filename)
    return data["x"], data["y"]


def load_features_as_dataframe(filename):
    x, y = load_features(rootpath / filename)
    df = pd.DataFrame(x)
    df["y"] = pd.DataFrame({"y": y})
    return df


def save_features(cls, filename, x, y):
    np.savez_compressed(rootpath / filename, x=x, y=y)


def extract_features(model, datagen, filename):
    filepath = rootpath / filename
    if not filepath.exists():
        datagen.reset()
        x = np.array([])
        y = np.array([])
        x = x.reshape((0, 1280))
        while x.shape[0] < datagen.n:
            im, label = datagen.next()
            yi = label.argmax(axis=1)
            xi = model(im).numpy()
            y = np.hstack((y, yi))
            x = np.vstack((x, xi))
            print(f"{x.shape[0]} images processed on {filename}")
            clear_output(wait=True)
        save_features(filename, x, y)
    else:
        print(f"{filename:30} already exists")


def evaluate_models(train_path, val_path):
    train_df = load_features_as_dataframe(train_path)
    val_df = load_features_as_dataframe(val_path)

    setup(data=train_df, target="y", test_data=val_df, silent=True)
    compare_models(cross_validation=False)

### Redes Neurais Convolucionais

In [5]:
class TransferNet(tf.keras.Model):
    def __init__(self, convnet):
        super(TransferNet, self).__init__()
        self.conv = convnet
        self.pool = tf.keras.layers.GlobalAvgPool2D()
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, inputs):
        x = self.conv(inputs)
        x = self.pool(x)
        x = self.flatten(x)
        return x

### Transfer Learning com EfficientNet

#### EfficientNet B0

In [ ]:
# b0 = TransferNet(
#     tf.keras.applications.EfficientNetB0(
#         include_top=False,
#         weights='imagenet',
#     )
# )

extract_features(b0, train_generator, "efficientnetb0-train.npz")
extract_features(b0, val_generator, "efficientnetb0-val.npz")

evaluate_models("efficientnetb0-train.npz", "efficientnetb0-val.npz")

Description             Value
0                               session_id              5314
1                                   Target                 y
2                              Target Type        Multiclass
3                            Label Encoded              None
4                            Original Data     (44555, 1281)
5                           Missing Values             False
6                         Numeric Features              1280
[... 44 linhas omitidas na versão pública ...]
51                       Feature Selection             False
52                Feature Selection Method           classic
53            Features Selection Threshold              None
54                     Feature Interaction             False
55                           Feature Ratio             False
56                   Interaction Threshold              None
57                           Fix Imbalance             False
58                    Fix Imbalance Method             SMOTE

IntProgress(value=0, description='Processing: ', max=69)

\
                                                 
Initiated  . . . . . . . . . . . . . . . . . .   
Status     . . . . . . . . . . . . . . . . . .   
Estimator  . . . . . . . . . . . . . . . . . .   

                                                 
                                                 
Initiated                              22:41:52  
Status     Fitting Gradient Boosting Classifier  
Estimator          Gradient Boosting Classifier

Model  Accuracy     AUC  Recall   Prec.  \
qda    Quadratic Discriminant Analysis    0.6298  0.7641  0.4100  0.5979   
rf            Random Forest Classifier    0.5942  0.7240  0.3710  0.5171   
knn             K Neighbors Classifier    0.5049  0.6435  0.3337  0.4662   
ada               Ada Boost Classifier    0.4746  0.5913  0.2975  0.4191   
dt            Decision Tree Classifier    0.4425  0.5871  0.3160  0.4556   
lr                 Logistic Regression    0.3882  0.0000  0.2500  0.1507   
svm                SVM - Linear Kernel    0.3882  0.0000  0.2500  0.1507   
ridge                 Ridge Classifier    0.3882  0.0000  0.2500  0.1507   
nb                         Naive Bayes    0.2768  0.5585  0.2692  0.4029   

           F1   Kappa     MCC  TT (Sec)  
qda    0.5811  0.3885  0.4102     18.70  
rf     0.5389  0.3218  0.3328    140.67  
knn    0.4810  0.2030  0.2055     10.77  
ada    0.4112  0.1276  0.1486    277.79  
dt     0.4483  0.1570  0.1573    143.20  
lr     0.2171  0.0000  0.0000      2.74  
svm    0.2171  0.0000  0.0000     11.80  
ridge  0.2171  0.0000  0.0000      1.56  
nb     0.3006  0.0413  0.0452      0.67

#### EfficientNet B4

In [ ]:
# b4 = TransferNet(
#     tf.keras.applications.EfficientNetB4(
#         include_top=False,
#         weights='imagenet',
#     )
# )
# b4.extract(train_generator, "efficientnetb4-train.npz")
# b4.extract(val_generator, "efficientnetb4-val.npz")

In [ ]:
# TODO: Avaliar features

### ResNet 50

In [ ]:
# resnet50 = TransferNet(
#     tf.keras.applications.ResNet50(
#         include_top=False,
#         weights='imagenet',
#     )
# )
# resnet50.extract(train_generator, "resnet50-train.npz")
# resnet50.extract(val_generator, "resnet50-val.npz")

### VGG16

In [ ]:
# vgg16 = TransferNet(
#     tf.keras.applications.VGG16(
#         include_top=False,
#         weights='imagenet',
#     )
# )
# vgg16.extract(train_generator, "vgg16-train.npz")
# vgg16.extract(val_generator, "vgg16-val.npz")

In [ ]:
# TODO: Avaliar features

## Experimentos com Transformers

## Experimentos com Autoencoders

### Explicabilidade com CAM ou Grad-CAM